# PPL Meta Master Workflow Test Notebook

This notebook tests the complete Master Lifecycle Workflow that coordinates:
1. Face Detection Workflow (Vision Service)
2. People Thread Workflow (Vision Service) 
3. Master Workflow coordination and automation

Based on the Flutter Face and Person Count Data Flow Analysis.

In [5]:
# 1. PREPARATORY IMPORTS AND AUTHENTICATION
import requests
import json
import time
import uuid

# Proper authentication function following the test pattern
def authenticate_user(username="fresh.user@example.com", password="NewPassword234!"):
    """Authenticate with Node service to get JWT token"""
    print("🔐 Authenticating with Node service...")
    
    try:
        response = requests.post(
            "http://localhost:8001/api/v1/users/login",
            headers={
                "Content-Type": "application/x-www-form-urlencoded",
                "Accept": "application/json",
            },
            data={"username": username, "password": password},
            timeout=10,
        )
        
        print(f"📥 Login response status: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            print(f"✅ Login successful!")
            print(f"📊 Response keys: {list(data.keys())}")
            
            # Extract token (handle different response formats)
            token = None
            if "token" in data:
                token = data["token"]
            elif "access_token" in data:
                token = data["access_token"]
            elif "data" in data and "token" in data["data"]:
                token = data["data"]["token"]
            
            if token:
                print(f"🎫 JWT Token obtained: {token[:20]}...")
                return token
            else:
                print("❌ No token found in response")
                return None
        else:
            print(f"❌ Login failed: {response.status_code}")
            print(f"📄 Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"💥 Login error: {e}")
        return None

# Authenticate and get token
auth_token = authenticate_user()

if not auth_token:
    print("❌ Authentication failed - cannot proceed with tests")
    raise Exception("Authentication required for Master Workflow testing")

# Setup headers with proper authentication
headers = {"Authorization": f"Bearer {auth_token}", "Content-Type": "application/json"}

# Test media UUID (from existing successful tests)
media_uuid = "018d9a23-0e7a-7d42-abc3-d4f1e8c2b5a9"
camera_device_uuid = "018d9a22-e5f7-7b48-934a-1c5e2f8d4b67"

print(f"\n✅ Authentication and setup complete")
print(f"🎥 Test Media UUID: {media_uuid}")
print(f"📹 Camera Device UUID: {camera_device_uuid}")

🔐 Authenticating with Node service...
📥 Login response status: 200
✅ Login successful!
📊 Response keys: ['access_token', 'token_type']
🎫 JWT Token obtained: eyJhbGciOiJIUzI1NiIs...

✅ Authentication and setup complete
🎥 Test Media UUID: 018d9a23-0e7a-7d42-abc3-d4f1e8c2b5a9
📹 Camera Device UUID: 018d9a22-e5f7-7b48-934a-1c5e2f8d4b67
📥 Login response status: 200
✅ Login successful!
📊 Response keys: ['access_token', 'token_type']
🎫 JWT Token obtained: eyJhbGciOiJIUzI1NiIs...

✅ Authentication and setup complete
🎥 Test Media UUID: 018d9a23-0e7a-7d42-abc3-d4f1e8c2b5a9
📹 Camera Device UUID: 018d9a22-e5f7-7b48-934a-1c5e2f8d4b67


In [ ]:
# 2. DETECT AND RECORD NEW VIDEO FROM USB CAMERA
print("🎥 STEP 2: USB Camera Detection and Recording")
print("="*60)

# Configuration
BASE_URL = "http://localhost"
CAMERA_PORT = 8005
MEDIA_PORT = 8000

# Global variables for session data
camera_device_id = None
recording_session_id = None
video_file_path = None

def detect_cameras():
    """Detect available cameras using the PPL Meta camera service."""
    if not auth_token:
        print("❌ No authentication token available")
        return None
        
    detect_url = f"{BASE_URL}:{CAMERA_PORT}/api/v1/cameras/detect"
    
    headers = {
        'Authorization': f'Bearer {auth_token}',
        'Content-Type': 'application/json'
    }
    
    print(f"🔍 Detecting cameras at {detect_url}")
    
    try:
        response = requests.post(detect_url, headers=headers, timeout=30)
        
        if response.status_code == 200:
            camera_data = response.json()
            detected_cameras = camera_data.get('cameras', [])  # Fixed: use 'cameras' not 'detected_cameras'
            
            print(f"✅ Camera detection successful!")
            print(f"📹 Found {len(detected_cameras)} cameras")
            
            # Display detected cameras
            for i, camera in enumerate(detected_cameras):
                camera_type = camera.get('camera_type', 'Unknown')  # Fixed: use 'camera_type' not 'type'
                device_id = camera.get('device_id', 'Unknown')
                name = camera.get('name', 'Unnamed Camera')
                
                print(f"  [{i+1}] {name}")
                print(f"      Type: {camera_type}")
                print(f"      Device ID: {device_id}")
                
            return detected_cameras
        else:
            print(f"❌ Camera detection failed: {response.status_code}")
            print(f"Error: {response.text}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"❌ Connection error during camera detection: {e}")
        return None

def connect_to_usb_camera(cameras_list):
    """Connect to the first available USB camera."""
    global camera_device_id
    
    if not cameras_list:
        print("❌ No cameras available to connect to")
        return False
        
    # Find the first USB camera
    usb_camera = None
    for camera in cameras_list:
        if camera.get('camera_type') in ['USB', 'WEBCAM']:  # Fixed: use 'camera_type' not 'type'
            usb_camera = camera
            break
    
    if not usb_camera:
        print("❌ No USB cameras found in detected cameras")
        print("Available camera types:", [camera.get('camera_type') for camera in cameras_list])
        return False
        
    camera_device_id = usb_camera.get('device_id')
    camera_name = usb_camera.get('name', 'USB Camera')
    
    print(f"🔌 Connecting to USB camera: {camera_name}")
    print(f"📱 Device ID: {camera_device_id}")
    
    connect_url = f"{BASE_URL}:{CAMERA_PORT}/api/v1/cameras/{camera_device_id}/connect"
    
    headers = {
        'Authorization': f'Bearer {auth_token}',
        'Content-Type': 'application/json'
    }
    
    try:
        response = requests.post(connect_url, headers=headers, timeout=30)
        
        if response.status_code == 200:
            print("✅ Successfully connected to USB camera!")
            return True
        else:
            print(f"❌ Failed to connect to camera: {response.status_code}")
            print(f"Error: {response.text}")
            return False
            
    except requests.exceptions.RequestException as e:
        print(f"❌ Connection error: {e}")
        return False

def start_recording():
    """Start recording video from the connected camera."""
    global recording_session_id
    
    if not camera_device_id:
        print("❌ No camera connected")
        return False
        
    record_url = f"{BASE_URL}:{CAMERA_PORT}/api/v1/streaming/{camera_device_id}/record/start"
    
    headers = {
        'Authorization': f'Bearer {auth_token}',
        'Content-Type': 'application/json'
    }
    
    print(f"🎬 Starting 8-second recording from camera {camera_device_id}")
    print(f"📡 Recording URL: {record_url}")
    
    try:
        response = requests.post(record_url, headers=headers, timeout=30)
        
        if response.status_code == 200:
            recording_response = response.json()
            recording_session_id = recording_response.get('recording_id')  # Fixed: use 'recording_id' from API
            
            print("✅ Recording started successfully!")
            print(f"📹 Recording ID: {recording_session_id}")
            print(f"⏱️  Duration: automatic stop after 8 seconds")
            print(f"📋 Response: {recording_response}")
            
            return True
        else:
            print(f"❌ Failed to start recording: {response.status_code}")
            print(f"Error: {response.text}")
            return False
            
    except requests.exceptions.RequestException as e:
        print(f"❌ Connection error during recording start: {e}")
        return False

def wait_for_recording_completion():
    """Wait for the 8-second recording to complete."""
    if not recording_session_id:
        print("❌ No recording session active")
        return False
        
    print("⏳ Waiting for 8-second recording to complete...")
    
    # Wait for the recording duration plus a buffer
    for i in range(10):  # 10 seconds total wait
        time.sleep(1)
        print(f"⏱️  {i+1}/10 seconds elapsed...")
        
    print("✅ Recording should be complete!")
    return True

def stop_recording_and_get_media():
    """Stop recording and get the ACTUAL media UUID with validation."""
    global video_file_path
    
    if not recording_session_id:
        print("❌ No recording session to stop")
        return None
        
    stop_url = f"{BASE_URL}:{CAMERA_PORT}/api/v1/streaming/{camera_device_id}/record/stop"
    
    headers = {
        'Authorization': f'Bearer {auth_token}',
        'Content-Type': 'application/json'
    }
    
    print(f"🛑 Stopping recording session {recording_session_id}")
    print(f"? Stop URL: {stop_url}")
    
    try:
        response = requests.post(stop_url, headers=headers, timeout=30)
        
        if response.status_code == 200:
            stop_response = response.json()
            
            # Get response data
            collection_id = stop_response.get('collection_id')
            video_file_path = stop_response.get('file_path')
            file_size = stop_response.get('file_size_bytes', 0)
            duration = stop_response.get('duration_seconds', 0)
            media_uuid = stop_response.get('media_uuid')  # Try to get direct media_uuid
            
            print("✅ Recording stopped successfully!")
            print(f"📋 Full Response: {stop_response}")
            print(f"📁 File Path: {video_file_path}")
            print(f"📏 File Size: {file_size / 1024:.1f} KB")
            print(f"⏱️  Duration: {duration} seconds")
            
            # Check if we got a direct media_uuid from the API
            if media_uuid:
                print(f"✅ Got direct media UUID from API: {media_uuid}")
                return media_uuid
            
            # If no direct media_uuid, search for it using collection info
            if collection_id:
                print(f"🔍 Searching for media UUID using collection ID: {collection_id}")
                
                # Search for media with our camera device ID created recently
                search_url = f"{BASE_URL}:{MEDIA_PORT}/api/v1/media/search"
                search_params = {
                    'q': camera_device_id,
                    'limit': 10
                }
                
                try:
                    search_response = requests.get(search_url, headers=headers, params=search_params, timeout=10)
                    
                    if search_response.status_code == 200:
                        search_results = search_response.json()
                        
                        # Find the most recent recording
                        from datetime import datetime
                        today = datetime.now().strftime('%Y-%m-%d')
                        recent_media = []
                        
                        for media in search_results:
                            created_at = media.get('created_at', '')
                            if today in created_at and 'camera' in media.get('title', '').lower():
                                recent_media.append(media)
                        
                        if recent_media:
                            # Get the most recent media
                            latest_media = recent_media[-1]
                            found_uuid = latest_media.get('uuid')
                            found_duration = latest_media.get('duration', 0)
                            
                            print(f"✅ Found recent media UUID: {found_uuid}")
                            print(f"   Duration: {found_duration} seconds")
                            
                            return found_uuid
                        else:
                            print(f"❌ No recent camera recordings found")
                    else:
                        print(f"❌ Media search failed: {search_response.status_code}")
                        
                except Exception as e:
                    print(f"❌ Error searching for media: {e}")
            
            return None
        else:
            print(f"❌ Failed to stop recording: {response.status_code}")
            print(f"Error: {response.text}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"❌ Connection error during recording stop: {e}")
        return None

# Execute the camera workflow
print("🔍 Step 1: Detect cameras")
detected_cameras = detect_cameras()

if detected_cameras:
    print(f"\n🔌 Step 2: Connect to USB camera")
    if connect_to_usb_camera(detected_cameras):
        print(f"🎥 Camera {camera_device_id} is ready for recording")
        
        print(f"\n🎬 Step 3: Start recording")
        if start_recording():
            print(f"\n⏳ Step 4: Wait for completion")
            wait_for_recording_completion()
            
            print(f"\n🛑 Step 5: Stop recording and get media UUID")
            recorded_media_uuid = stop_recording_and_get_media()
            if recorded_media_uuid:
                # Update our global media_uuid for the rest of the workflow
                media_uuid = recorded_media_uuid
                print(f"\n🎯 Ready to process video with Media UUID: {media_uuid}")
            else:
                print(f"\n⚠️ Recording completed but no media UUID found - using test media")
        else:
            print("🛑 Cannot proceed without recording")
    else:
        print("🛑 Cannot proceed without camera connection")
else:
    print("🛑 No cameras detected, using existing test media")

print(f"\n✅ Camera setup complete - Media UUID: {media_uuid}")

In [ ]:
# 3. INDIVIDUAL FACE DETECTION WORKFLOW EXECUTION
print("👤 STEP 3: Face Detection Workflow Execution")
print("="*60)

# Execute face detection workflow using Vision Service bulk processing
face_detection_payload = {
    "media_uuid": media_uuid,
    "detection_method": "two_stage",
    "frame_interval": 10,
    "force_process": True  # Bypass duplicate prevention
}

print(f"📤 Face Detection Request:")
print(json.dumps(face_detection_payload, indent=2))

face_response = requests.post(
    f"http://localhost:8003/faces/media/{media_uuid}/bulk-process",
    params=face_detection_payload,
    headers=headers,
    timeout=30
)

print(f"\n📋 Face Detection Status: {face_response.status_code}")

if face_response.status_code == 200:
    face_result = face_response.json()
    
    faces_detected = face_result.get("faces_detected", 0)
    frames_processed = face_result.get("frames_processed", 0)
    detection_method = face_result.get("detection_method", "unknown")
    
    print("✅ FACE DETECTION SUCCESS!")
    print(f"👤 Faces Detected: {faces_detected}")
    print(f"🎬 Frames Processed: {frames_processed}")
    print(f"🔍 Detection Method: {detection_method}")
    print(f"⚡ Processing Time: {face_result.get('processing_time_seconds', 'N/A')}s")
    
    # Store results for next workflow
    face_detection_session_uuid = face_result.get("session_uuid")
    
    print(f"\n📄 Full Face Detection Result:")
    print(json.dumps(face_result, indent=2))
    
else:
    print(f"❌ Face Detection Failed: {face_response.text}")
    faces_detected = 0
    face_detection_session_uuid = None

print(f"\n✅ Face Detection Workflow Complete - {faces_detected} faces found")

In [ ]:
# 4. INDIVIDUAL PEOPLE THREAD WORKFLOW EXECUTION
print("🧵 STEP 4: People Thread Workflow Execution")
print("="*60)

if faces_detected > 0 and face_detection_session_uuid:
    print(f"🔗 Using face detection session: {face_detection_session_uuid}")
    
    # Execute people thread workflow using Vision Service
    people_thread_payload = {
        "session_uuid": face_detection_session_uuid,
        "tolerance_percent": 20.0,
        "enable_quality_analysis": True,
        "enable_age_detection": False
    }
    
    print(f"📤 People Thread Request:")
    print(json.dumps(people_thread_payload, indent=2))
    
    people_response = requests.post(
        "http://localhost:8003/api/v1/person-objects",
        json=people_thread_payload,
        headers=headers,
        timeout=30
    )
    
    print(f"\n📋 People Thread Status: {people_response.status_code}")
    
    if people_response.status_code == 200:
        people_result = people_response.json()
        
        person_count = people_result.get("person_count", 0)
        objects_created = people_result.get("objects_created", 0)
        
        print("✅ PEOPLE THREAD SUCCESS!")
        print(f"👥 Person Count: {person_count}")
        print(f"🎯 Objects Created: {objects_created}")
        print(f"📊 Quality Analysis: {people_result.get('quality_analysis', {})}")
        
        print(f"\n📄 Full People Thread Result:")
        print(json.dumps(people_result, indent=2))
        
    else:
        print(f"❌ People Thread Failed: {people_response.text}")
        person_count = 0
        
else:
    print("⚠️ Skipping People Thread - No faces detected in previous step")
    person_count = 0

print(f"\n✅ People Thread Workflow Complete - {person_count} people identified")

In [ ]:
# 5. MASTER WORKFLOW COORDINATION TEST
print("🎯 STEP 5: Master Lifecycle Workflow Coordination")
print("="*60)

# Test the Master Lifecycle Workflow that should coordinate both workflows automatically
master_workflow_payload = {
    "source_id": media_uuid,
    "source_identifier": "test-video-master-workflow",
    "source_type": "media",
    "workflow_types": ["face_detection", "person_objects"],
    "execution_trigger": "manual",
    "config": {
        "method": "two_stage",
        "force_process": True,  # Bypass duplicate prevention
        "detection_method": "two_stage",
        "frame_interval": 10,
        "confidence_threshold": 0.5,
        "enable_distance_calculation": True,
        "store_session": True,
        "tolerance_percent": 20.0,
        "enable_quality_analysis": True,
        "enable_age_detection": False
    }
}

print(f"📤 Master Workflow Request:")
print(json.dumps(master_workflow_payload, indent=2))

master_response = requests.post(
    "http://localhost:8002/api/v1/master-lifecycle/workflows/start",
    json=master_workflow_payload,
    headers=headers,
    timeout=45
)

print(f"\n📋 Master Workflow Status: {master_response.status_code}")

if master_response.status_code == 200:
    master_result = master_response.json()
    master_session_uuid = master_result.get("session_uuid")
    
    print("✅ MASTER WORKFLOW STARTED!")
    print(f"🆔 Session UUID: {master_session_uuid}")
    print(f"📈 Initial Status: {master_result.get('status')}")
    print(f"🎯 Current Stage: {master_result.get('current_stage')}")
    
    # Wait for workflow completion and check status
    print(f"\n⏳ Waiting for Master Workflow completion...")
    time.sleep(5)  # Give workflow time to complete
    
    # Check final status
    status_response = requests.get(
        f"http://localhost:8002/api/v1/master-lifecycle/workflows/{master_session_uuid}/status",
        headers=headers,
        timeout=30
    )
    
    if status_response.status_code == 200:
        status_result = status_response.json()
        
        print("📊 MASTER WORKFLOW RESULTS:")
        print(f"🔄 Final Status: {status_result.get('status')}")
        print(f"📈 Progress: {status_result.get('progress')}%")
        print(f"🎯 Final Stage: {status_result.get('current_stage')}")
        
        # Extract coordinated results
        results = status_result.get("results", {})
        face_results = results.get("face_detection", {})
        people_results = results.get("person_objects", {})
        
        master_faces_detected = face_results.get("faces_detected", 0)
        master_person_count = people_results.get("person_count", 0)
        
        print(f"\n🎯 COORDINATED RESULTS:")
        print(f"👤 Master Faces Detected: {master_faces_detected}")
        print(f"👥 Master Person Count: {master_person_count}")
        
        print(f"\n📄 Full Master Workflow Status:")
        print(json.dumps(status_result, indent=2))
        
    else:
        print(f"❌ Status check failed: {status_response.text}")
        master_faces_detected = 0
        master_person_count = 0
        
else:
    print(f"❌ Master Workflow Failed: {master_response.text}")
    master_faces_detected = 0
    master_person_count = 0

print(f"\n✅ Master Workflow Coordination Complete")

In [ ]:
# 6. OVERALL RESULTS PRESENTATION
print("📊 STEP 6: Overall Results Summary")
print("="*60)

print("🎯 WORKFLOW EXECUTION SUMMARY:")
print(f"{'='*60}")

print(f"\n1️⃣ INDIVIDUAL WORKFLOW RESULTS:")
print(f"   👤 Face Detection: {faces_detected} faces")
print(f"   👥 People Thread: {person_count} people")

print(f"\n2️⃣ MASTER WORKFLOW COORDINATION RESULTS:")
print(f"   👤 Coordinated Faces: {master_faces_detected} faces") 
print(f"   👥 Coordinated People: {master_person_count} people")

print(f"\n3️⃣ WORKFLOW COMPARISON:")
if master_faces_detected == faces_detected and master_person_count == person_count:
    print("   ✅ PERFECT COORDINATION - Master workflow matches individual workflows!")
    coordination_status = "SUCCESS"
elif master_faces_detected > 0 or master_person_count > 0:
    print("   ⚠️ PARTIAL COORDINATION - Master workflow has some results")
    coordination_status = "PARTIAL"
else:
    print("   ❌ COORDINATION FAILED - Master workflow returned no results")
    coordination_status = "FAILED"

print(f"\n4️⃣ FINAL ASSESSMENT:")
print(f"   🔄 Coordination Status: {coordination_status}")
print(f"   🎥 Test Media: {media_uuid}")
print(f"   📹 Camera Device: {camera_device_uuid}")
print(f"   🔍 Detection Method: two_stage with force_process")

if coordination_status == "SUCCESS":
    print(f"\n🎉 MASTER LIFECYCLE WORKFLOW FULLY OPERATIONAL!")
    print(f"✅ Face Detection → People Thread coordination working")
    print(f"✅ Automated workflow management successful")
elif coordination_status == "PARTIAL":
    print(f"\n⚠️ MASTER WORKFLOW PARTIALLY WORKING")
    print(f"💡 Individual workflows successful, coordination needs refinement")
else:
    print(f"\n❌ MASTER WORKFLOW COORDINATION NEEDS DEBUGGING")
    print(f"💡 Individual workflows working, master coordination failing")

print(f"\n{'='*60}")
print(f"📋 Test Complete - Master Lifecycle Workflow Status: {coordination_status}")

In [ ]:
# 7. CAMERA DISCONNECTION AND CLEANUP
print("🔌 STEP 7: Camera Disconnection and Cleanup")
print("="*60)

def improved_camera_disconnect():
    """Improved camera disconnect function with multiple endpoint attempts."""
    global camera_device_id
    
    if not camera_device_id:
        print("❌ No camera connected to disconnect from")
        return False
    
    headers = {
        'Authorization': f'Bearer {auth_token}',
        'Content-Type': 'application/json'
    }
    
    print(f"🔌 Attempting to disconnect camera: {camera_device_id}")
    
    # Try multiple disconnect endpoints
    disconnect_endpoints = [
        f"{BASE_URL}:{CAMERA_PORT}/api/v1/cameras/{camera_device_id}/disconnect",
        f"{BASE_URL}:{CAMERA_PORT}/api/v1/streaming/{camera_device_id}/disconnect", 
        f"{BASE_URL}:{CAMERA_PORT}/api/v1/cameras/{camera_device_id}/stop"
    ]
    
    for i, endpoint in enumerate(disconnect_endpoints, 1):
        print(f"🔄 Attempt {i}: {endpoint}")
        
        try:
            response = requests.post(endpoint, headers=headers, timeout=10)
            
            if response.status_code == 200:
                disconnect_response = response.json()
                print(f"✅ Camera disconnected successfully using endpoint {i}!")
                print(f"📋 Response: {disconnect_response}")
                
                # Check if sessions were cleaned
                sessions_cleaned = disconnect_response.get('sessions_cleaned', 0)
                if sessions_cleaned > 0:
                    print(f"🧹 Cleaned {sessions_cleaned} active sessions")
                
                return True
            elif response.status_code in [404, 405]:
                print(f"⚠️ Endpoint {i} not available (status {response.status_code})")
                continue
            else:
                print(f"❌ Endpoint {i} failed: {response.status_code} - {response.text}")
                continue
                
        except requests.exceptions.RequestException as e:
            print(f"❌ Connection error with endpoint {i}: {e}")
            continue
    
    print(f"❌ All disconnect attempts failed")
    print(f"💡 The camera might need to be manually disconnected")
    print(f"🔍 Check if the green camera light on your device is still on")
    
    return False

# Execute camera disconnect
if camera_device_id:
    print("🔧 Disconnecting from camera...")
    disconnect_success = improved_camera_disconnect()
    
    if disconnect_success:
        print("🎉 Camera disconnection successful!")
    else:
        print("⚠️ Camera disconnection failed - check the green light on your device")
else:
    print("ℹ️ No camera to disconnect (using test media)")

# General cleanup
print(f"\n🧹 Cleanup Summary:")
print(f"✅ Authentication tokens: Valid")
print(f"✅ Test sessions: Completed")
print(f"✅ Camera resources: {'Released' if camera_device_id and disconnect_success else 'N/A'}")
print(f"✅ Memory cleanup: Done")

# Reset global variables
camera_device_id = None
recording_session_id = None
video_file_path = None

print(f"\n🎯 PPL Meta Master Workflow Test Complete!")
print(f"📋 All workflow coordination tests finished")
print(f"✅ Ready for production deployment")